# π0.5 × LIBERO-plus LoRA — PARC2026 予選 track1 向け（モデル乗り換え1回きり）

SmolVLA(0.5B) から **π0.5(3B, PaliGemma バックボーン)** へ乗り換える。

## なぜ乗り換えるのか

推論側（chunk長・平滑化・投票・RTC）と学習側（データ量・タスク種・画像拡張・重複排除）を
一通り試し、公式スコアは **0.13178 で頭打ち**。直交実験でボトルネックが判明した:

```
12回すべて done=True（タスク自体は解けている）
成功は4回。残り8回は「対象外の物体を1mm動かした」判定で無効
```

**能力ではなく精度**が足りない。→ より強いバックボーンに賭ける。

## この乗り換えの制約（先に読むこと）

| 項目 | 内容 |
|---|---|
| 重み | **7.5 GB**（SmolVLA は 0.9 GB）。zip上限20GBには収まる |
| 採点環境 | L4 24GB。bf16 で載る |
| **ローカル評価** | **不可能**。母艦のRAMが7.4GB、Dockerに4GBしか割けない |
| → 測定手段 | **このノートブックのセクション11-13だけ**。必ず実行すること |
| PaliGemma | **ゲート付きリポジトリ**。非ゲートのミラーを使う（語彙数を検証済み） |
| 学習GPU | A100 推奨。L4/T4 では `train_expert_only` でも苦しい |

⚠️ **ローカルで測れない以上、セクション12の比較評価を飛ばしたら
提出は完全な博打になる。** SmolVLA の時に飛ばして半日を失った工程がこれ。


## 1. ランタイムを確認する

In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_LEROBOT_HOME"] = "/content/lerobot_cache"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12 以上が必要です。")

if not torch.cuda.is_available():
    raise RuntimeError("GPU ランタイムを選択してください。")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU     : {GPU_NAME}")
print(f"VRAM    : {GPU_GIB:.1f} GiB")
print(f"bfloat16: {'対応' if torch.cuda.is_bf16_supported() else '非対応（float16 で回す）'}")

## 2. システムパッケージを準備する

In [ ]:
def run_quiet(command: list[str], *, check: bool = True) -> subprocess.CompletedProcess:
    result = subprocess.run(
        command, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False,
    )
    if check and result.returncode != 0:
        raise RuntimeError(result.stdout[-6000:])
    return result


run_quiet(["apt-get", "update", "-qq"])
run_quiet([
    "apt-get", "install", "-y", "-qq",
    "ffmpeg", "git", "unzip", "libgl1", "libglib2.0-0",
    "libsm6", "libxext6", "libexpat1",
    "libfontconfig1-dev", "libmagickwand-dev",
])

print("System packages ready.")

## 3. LeRobot を導入する

`v0.6.0` を固定。**`torchao` は必ず消す**（SmolVLA のインポートと衝突する）。

In [ ]:
LEROBOT_TAG = "v0.6.0"
LEROBOT_DIR = Path("/content/lerobot")
LEROBOT_SRC = LEROBOT_DIR / "src"

run_quiet([sys.executable, "-m", "pip", "uninstall", "-y", "lerobot", "torchao"], check=False)
shutil.rmtree(LEROBOT_DIR, ignore_errors=True)

run_quiet([
    "git", "clone", "--quiet", "--depth", "1", "--branch", LEROBOT_TAG,
    "https://github.com/huggingface/lerobot.git", str(LEROBOT_DIR),
])

# bf16 非対応 GPU（T4 など）では fp16 に落とす
smolvlm_source = LEROBOT_SRC / "lerobot" / "policies" / "smolvla" / "smolvlm_with_expert.py"
if not torch.cuda.is_bf16_supported():
    source = smolvlm_source.read_text(encoding="utf-8")
    source = source.replace('torch_dtype="bfloat16",', 'torch_dtype="float16",', 1)
    smolvlm_source.write_text(source, encoding="utf-8")

# 学習ログを静かにする
train_script = LEROBOT_SRC / "lerobot" / "scripts" / "lerobot_train.py"
source = train_script.read_text(encoding="utf-8")
source = source.replace("logging.info(pformat(cfg.to_dict()))", "logging.debug(pformat(cfg.to_dict()))", 1)
source = source.replace("disable=inside_slurm(),", "disable=True,", 1)
train_script.write_text(source, encoding="utf-8")

run_quiet([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
           "-e", f"{LEROBOT_DIR}[training,smolvla,peft]"])
run_quiet([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

for module_name in list(sys.modules):
    if module_name.split(".")[0] in {"lerobot", "torchao"}:
        del sys.modules[module_name]

sys.path = [item for item in sys.path if item not in {str(LEROBOT_DIR), str(LEROBOT_SRC)}]
sys.path.insert(0, str(LEROBOT_SRC))
importlib.invalidate_caches()

try:
    importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("torchao の削除に失敗しました。ランタイムを再起動してやり直してください。")

import lerobot
import peft

if LEROBOT_SRC.resolve() not in Path(lerobot.__file__).resolve().parents:
    raise RuntimeError("LeRobot の読込先が想定と違います。")

print("LeRobot ready.")

## 4. 設定 ← **ここが前回との違いの本体**

### 学習対象タスク

PARC2026 予選 track1 の4タスクは、基本タスクとしてはこの4種：

| # | track1 のタスク名 | 言語指示 |
|---|---|---|
| 0 | `..._black_bowl_in_the_top_drawer_..._table_2` | pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate |
| 1 | `..._tomato_sauce_..._basket_table_27` | pick up the tomato sauce and place it in the basket |
| 2 | `..._milk_..._basket_light_15` | pick up the milk and place it in the basket |
| 3 | `put_the_bowl_on_the_stove_light_11` | put the bowl on the stove |

**前回は「黒いボウル→皿」の10タスクだけを学習していた。** カゴもコンロも一度も見せていない。

### バッチサイズの自動調整

バッチ32はVRAM 40GB級が要る。小さいGPUでは自動で下げ、**総サンプル数 `TOTAL_SAMPLES` が一定になるようステップ数を増やす**。

In [ ]:
# ★ π0.5 の LIBERO 微調整済み重み。lerobot 配下で最もDLの多い LIBERO モデル。
#   pi05_libero_base（未微調整）ではなく finetuned を土台にする。
BASE_MODEL_REPO = "lerobot/pi05_libero_finetuned_v044"
BASE_MODEL_REVISION = "main"

# ★ PaliGemma のトークナイザ。
#   本家 google/paligemma-3b-pt-224 は **ゲート付き**でトークン無しでは 401。
#   非ゲートのミラーを使う。同一性は語彙数で検証済み:
#     lm_head の行数           257152 = 256000(base) + 1024(<loc>) + 128(<seg>)
#     ミラーの len(tokenizer)  257153 = 上記 + <image>(埋め込み専用、予測されない)
#   本家にアクセスできるなら VLM_REPO を差し替えてよい。
VLM_REPO = "leo009/paligemma-3b-pt-224"

DATASET_REPO = "lerobot/libero_plus"
DATASET_REVISION = "f3f49f426d75030177b18778374005bc12ccd588"

# ---- track1 の4タスク種（言語指示。大文字小文字・記号は正規化して照合する）----
TARGET_TASK_NAMES = [
    "pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate",
    "pick up the tomato sauce and place it in the basket",
    "pick up the milk and place it in the basket",
    "put the bowl on the stove",
]

# ★ 学習に使うタスクの範囲
#   4種だけを厚く学習しても未知タスクには効かない。データセットの全タスクを
#   使い、物体・配置・言い回しの多様性そのものを増やす。
# ⚠️ 2026-08-09 実測: 全40タスクへ広げる（v4）は公式 0.13178 → 0.06948 と半減した。
#    v4 では track1 の4タスクが 60話→10話（6分の1）に減っていた。
#      多様性はタスク種ではなく、**同じタスクの摂動変種**で稼ぐべき。
USE_ALL_TASKS = False                 # v4 で否定された。4種に戻す
TRAIN_EPISODES_PER_TASK = 60          # v1 と同じ本数

# ★ エピソードの選び方
#   'spread' … 全域から等間隔（v1 の方式）
#   'block'  … 連続ブロックから取る
#
#   ⚠️ libero_plus は観測を動画(mp4)で持つ。全域に散らばった240話に触ると
#      動画チャンクをほぼ全部落とすことになり、4時間経っても学習に入らなかった。
#   ★ 2026-08-10 実測: 摂動タイプは番号順にシャッフルされており、
#      等間隔60本の明度分布（平均・標準偏差・範囲）は全在庫とほぼ一致した。
#      → 連続ブロックでも代表性は同じ。落とすチャンク数だけが激減する。
EPISODE_PICK = "block"

# ★ 2026-08-09 の学習データ監査で判明したこと
#   v1 が学習した 240 エピソードのうち、**行動列がユニークなのは 124 本だけ**
#   （重複率 48.3%）。LIBERO-plus は1本のデモに複数の照明・テクスチャ摂動を
#   掛けて増やしており、エピソード数＝軌道の多様性ではない。
#   重複グループ内の画像は明度が大きく違う（中央 0.213）＝ light/table 系の摂動。
#   → 同じ把持軌道を2回暗記させる代わりに、別の軌道を入れたほうが情報量が多い。
# ★ 2026-08-10 判明: 重複排除は**逆向きの処理だった**。
#   「重複」の正体は、同じ行動軌道を別の照明・別のテクスチャで描き直したもの。
#   行動列の署名で削ると、1軌道あたりテクスチャ1種だけが残る＝視覚多様性を捨てる。
#   汎化テストの実測: light 摂動 100% / **table 摂動 25%**（幾何は完全同一、
#   BDDL のテクスチャ行が1行違うだけ）。削ったものと落ちている軸が一致する。
#   → 冗長ではなく多様性。減らすのではなく増やす側に倒す。
#
# ⚠️ 2026-08-09 実測: 重複排除した v6 はローカル4指標すべてで v1 に負けた
#    (成功 0.95→0.85 / 衝突 0.05→0.10 / ジャーク 15.88→16.36 / 軌道長 1.25→1.30)。
#    ただし TOTAL_SAMPLES 固定のまま母数を 240→178 に減らしたので、
#    1話あたりの周回数が1.35倍になる交絡があった。
#    **乗り換えの効果だけを見たいので、ここは v1 と同じ False に戻す。一度に2つ変えない。**
DEDUP_TRAJECTORIES = False            # ★ 下の注記のとおり、二度と True にしない

# ★ 取り落とし軌道の除外
#   グリッパの開閉が4回＝掴み直し＝一度落としている疑い。
#   ⚠️ ただし drawer 系は「取っ手を掴む→放す→ボウルを掴む→放す」で4回が正常。
#      単純な pick-and-place のみ除外する（tomato sauce が 60本中12本＝20%）。
DROP_REGRASP = False                  # 同上。v1 と条件を揃える

# ---- 学習ハイパーパラメータ ----
USE_AUGMENTATION = False              # ⚠️ 6種全部の拡張は実測で悪化した（上のコメント参照）
# ⚠️ v3 は「話数100」と「256kサンプル」を同時に変えて悪化した（ローカル 0.95→0.75）。
#    どちらが原因か切り分けていない。v5 では話数だけ動かし、サンプル数は v1 と同じに固定する。
# ⚠️ 3B のフル1周は高い。SmolVLA と同じ 128k サンプルだと A100 で 5〜8時間。
#    乗り換えが効くかを見るだけなら 48k で十分。効きそうなら増やす。
TOTAL_SAMPLES = 48_000
LEARNING_RATE = 1e-4                  # 前回 3e-4 → ベースモデルと同じ 1e-4 に
FINAL_LEARNING_RATE = 1e-5
WARMUP_STEPS = 500
LORA_R = 16
LORA_ALPHA = 16
SEED = 42
LOG_FREQ = 100

# ---- VRAM からバッチサイズを決め、ステップ数で補償する ----
# ⚠️ π0.5 は 3B。SmolVLA(0.5B) の 1/4 程度のバッチしか載らない。
#    train_expert_only=true で PaliGemma 本体は凍結し、
#    gemma_300m の action expert だけを学習する。
# ⚠️ 2026-08-10: はしごが 40GB 用のままで、A100 80GB でも batch 8 だった。
#    VRAM の1割しか使わず、時間単価の高い 80GB 機を無駄にしていた。
if GPU_GIB >= 70:
    BATCH_SIZE = 24       # A100 80GB
elif GPU_GIB >= 38:
    BATCH_SIZE = 8        # A100 40GB
elif GPU_GIB >= 20:
    BATCH_SIZE = 4        # L4 / A10 24GB
elif GPU_GIB >= 14:
    BATCH_SIZE = 2        # T4 16GB — 現実的でない。A100 を選び直すこと
else:
    BATCH_SIZE = 1

STEPS = max(1000, TOTAL_SAMPLES // BATCH_SIZE)
SAVE_FREQ = STEPS

MIXED_PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"

# ---- 比較評価の設定（セクション12で使う）----
EVAL_EPISODES_PER_TASK = 5    # 4タスク×5×2モデル=40話 ≈ 1時間
EVAL_N_ACTION_STEPS = 10      # ★ 提出時の設定(公式 0.13178)と揃える
EVAL_SEED = 2026

# ---- 出力先 ----
OUTPUT_DIR = Path("/content/outputs/pi05_libero_t1_lora")
MERGED_MODEL_DIR = Path("/content/pi05_libero_t1_lora_merged")
BASELINE_MODEL_DIR = Path("/content/pi05_libero_baseline")
BASE_EVAL_DIR = Path("/content/eval_base")
FINETUNED_EVAL_DIR = Path("/content/eval_finetuned")
COMPARISON_CSV_PATH = Path("/content/comparison.csv")
MERGED_ZIP_PATH = Path("/content/pi05_t1_lora_merged.zip")

print(f"モデル        : {BASE_MODEL_REPO}  (3B / 重み 7.5GB)")
print(f"GPU           : {GPU_NAME} ({GPU_GIB:.1f} GiB)")
print(f"batch_size    : {BATCH_SIZE}   (VRAM {GPU_GIB:.0f}GiB から決定)")
print(f"steps         : {STEPS:,}")
print(f"実効サンプル数: {BATCH_SIZE * STEPS:,}  ← 前回は 3,000")
print(f"learning_rate : {LEARNING_RATE}   ← 前回は 3e-4")
print(f"mixed_precision: {MIXED_PRECISION}")
print()
if GPU_GIB < 20:
    print("⚠️ π0.5(3B) には VRAM が足りません。A100 を選び直してください。")
    print("   このまま進めると OOM か、非現実的な所要時間になります。")
print()
print("⚠️ このモデルはローカル(RAM 4GB)で評価できません。")
print("   セクション12の比較評価が唯一の測定手段です。必ず実行してください。")

## 4.5 PaliGemma トークナイザを用意する — **π0.5 固有の関門**

本家 `google/paligemma-3b-pt-224` はゲート付きで 401 になる。
非ゲートのミラーを落として、LeRobot 側のハードコードを書き換える。


In [ ]:
# ★ PaliGemma トークナイザのゲート回避
#
#   lerobot/policies/pi05/processor_pi05.py は
#       tokenizer_name="google/paligemma-3b-pt-224"
#   をハードコードしている。この repo はゲート付きで、
#   HFトークン無しでは 401 GatedRepoError になり学習が始まらない。
#
#   → 非ゲートのミラーをローカルに落とし、その場所を指すよう書き換える。
#
# ⚠️ このセルはセクション5より前にあるので cached_or_downloaded_snapshot は未定義。
#    huggingface_hub を直接使う（数MBの初回取得だけなのでリトライ不要）。
import shutil
from pathlib import Path

from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
# ⚠️ 2026-08-10: ここに "!pip install --force-reinstall --no-deps transformers" を
#    書いていたが、これが lerobot.policies の import を壊し、
#      DecodingError: Couldn't find a choice class for 'pi05'
#    を引き起こした。依存関係を手で継ぎ足さないこと。
#    版がおかしいときはランタイムを作り直すほうが速い。
import transformers
print('transformers:', transformers.__version__, ' (lerobot v0.6.0 の要求 >=5.4,<5.6)')
TOKENIZER_DIR = Path("/content/paligemma_tokenizer")
if not TOKENIZER_DIR.is_dir():
    _d = snapshot_download(
        VLM_REPO,
        allow_patterns=["tokenizer*", "*special_tokens*",
                        "preprocessor_config.json", "processor_config.json"],
    )
    shutil.copytree(_d, TOKENIZER_DIR, dirs_exist_ok=True)
    shutil.rmtree(TOKENIZER_DIR / ".cache", ignore_errors=True)

_tk = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR))
print(f"トークナイザ  : {type(_tk).__name__}  語彙 {len(_tk):,}")
assert len(_tk) in (257152, 257153), f"語彙数が想定外: {len(_tk)}"

for _rel in ("policies/pi05/processor_pi05.py", "policies/pi0/processor_pi0.py"):
    _p = LEROBOT_SRC / "lerobot" / _rel
    if not _p.is_file():
        continue
    _t = _p.read_text(encoding="utf-8")
    if "google/paligemma-3b-pt-224" in _t:
        _p.write_text(_t.replace('"google/paligemma-3b-pt-224"',
                                 f'"{TOKENIZER_DIR}"'), encoding="utf-8")
        print(f"書き換え      : {_rel}")
    else:
        print(f"書き換え不要  : {_rel}")


# ★ 2026-08-10: ソース書き換えだけでは足りなかった
#   tokenizer_name は **チェックポイント同梱の policy_preprocessor.json** から読まれる。
#   lerobot-train は make_pi05_pre_post_processors のデフォルトを通らず、
#   PolicyProcessorPipeline.from_pretrained() が JSON の値をそのまま使う。
#   → 重みフォルダ側の JSON も書き換える必要がある。
GATED = "google/paligemma-3b-pt-224"


def patch_gated_tokenizer(model_dir) -> int:
    """重みフォルダ内の JSON に埋まっているゲート付きリポジトリ名を差し替える。"""
    model_dir = Path(model_dir)
    n = 0
    for f in model_dir.rglob("*.json"):
        try:
            t = f.read_text(encoding="utf-8")
        except (OSError, UnicodeDecodeError):
            continue
        if GATED in t:
            try:
                f.write_text(t.replace(GATED, str(TOKENIZER_DIR)), encoding="utf-8")
            except OSError as e:          # キャッシュが読み取り専用の場合
                print(f"    ⚠️ 書き込めない: {f.name}  {e}")
                continue
            n += 1
            print(f"    書き換え: {f.relative_to(model_dir)}")
    if n == 0:
        print(f"    （{model_dir.name}: 該当なし）")
    return n


## 5. HuggingFace からの取得を用意する（429 リトライ付き）

In [ ]:
import random
import time
from collections.abc import Callable
from typing import TypeVar

import httpx
from huggingface_hub import snapshot_download
from huggingface_hub.errors import HfHubHTTPError, LocalEntryNotFoundError

T = TypeVar("T")


def run_hf_with_retry(operation: Callable[[], T]) -> T:
    last_error: BaseException | None = None
    for attempt in range(6):
        try:
            return operation()
        except (HfHubHTTPError, httpx.HTTPStatusError) as error:
            last_error = error
            response = getattr(error, "response", None)
            status = getattr(response, "status_code", None)
            if status != 429 and "429" not in str(error):
                raise
            if attempt == 5:
                break
            headers = getattr(response, "headers", {}) or {}
            try:
                delay = float(headers.get("Retry-After", 15)) + 1
            except (TypeError, ValueError):
                delay = min(120, 15 * (2**attempt) + random.random())
            time.sleep(delay)
    raise RuntimeError("Hugging Face からの取得に失敗しました。") from last_error


def cached_or_downloaded_snapshot(
    repo_id: str, revision: str, *,
    allow_patterns: list[str] | None = None,
    ignore_patterns: list[str] | None = None,
) -> Path:
    try:
        return Path(snapshot_download(
            repo_id=repo_id, revision=revision, token=False,
            allow_patterns=allow_patterns, ignore_patterns=ignore_patterns,
            local_files_only=True,
        ))
    except (LocalEntryNotFoundError, FileNotFoundError):
        return Path(run_hf_with_retry(lambda: snapshot_download(
            repo_id=repo_id, revision=revision, token=False,
            allow_patterns=allow_patterns, ignore_patterns=ignore_patterns,
            max_workers=1,
        )))


print("Download helpers ready.")

## 6. 学習データを選ぶ — track1 の4タスク種

データセットのタスク名と `TARGET_TASK_NAMES` を正規化して突き合わせる。
**見つからなければ候補を表示して止まる**（前回のように黙って進まない）。

In [ ]:
import difflib
import re
from collections import defaultdict

import numpy as np
import pandas as pd

from lerobot.datasets.dataset_metadata import LeRobotDatasetMetadata


def normalize_task_name(value: str) -> str:
    value = value.lower().replace("_", " ")
    value = re.sub(r"[^a-z0-9 ]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def task_name_from_cell(value) -> str:
    if isinstance(value, str):
        return value
    try:
        if len(value) > 0:
            return str(value[0])
    except TypeError:
        pass
    return str(value)


def choose_evenly_spaced(episode_indices: list[int], count: int) -> list[int]:
    """均等に間引く。要求数がある分より多ければ全部返す。"""
    if count >= len(episode_indices):
        return list(episode_indices)
    positions = [round(i * (len(episode_indices) - 1) / (count - 1)) for i in range(count)]
    return [episode_indices[p] for p in sorted(set(positions))]


dataset_metadata = run_hf_with_retry(
    lambda: LeRobotDatasetMetadata(DATASET_REPO, revision=DATASET_REVISION)
)

task_to_episodes: dict[str, list[int]] = defaultdict(list)
for episode_index, task_cell in enumerate(dataset_metadata.episodes["tasks"]):
    task_to_episodes[task_name_from_cell(task_cell)].append(int(episode_index))

available_by_normalized = {normalize_task_name(t): t for t in task_to_episodes}
print(f"データセットのタスク数: {len(task_to_episodes)}")

selected_by_task: dict[str, list[int]] = {}
missing: list[str] = []

wanted: list[str] = []                       # track1 の4種は必ず入れる
for task_name in TARGET_TASK_NAMES:
    actual = available_by_normalized.get(normalize_task_name(task_name))
    if actual is None:
        missing.append(task_name)
    else:
        wanted.append(actual)

if USE_ALL_TASKS:
    for t in sorted(task_to_episodes):       # 残り全部を追加
        if t not in wanted:
            wanted.append(t)
    print(f"★ 全タスク学習: {len(wanted)} タスク")
else:
    print(f"track1 の {len(wanted)} タスクのみ")

# --- 行動統計から「軌道の署名」を作る -------------------------------------
#   検証済み: min/max/mean/std/count の連結ハッシュは、行動列そのものの
#   ハッシュと 100% 一致した（240話・124軌道で確認）。
#   これによりデータ本体（数百MB）を落とさずに重複を判定できる。
import hashlib

def _traj_signature(row) -> str:
    v = np.concatenate([np.asarray(row[f"stats/action/{k}"]).ravel()
                        for k in ("min", "max", "mean", "std", "count")])
    return hashlib.md5(np.round(v, 6).tobytes()).hexdigest()[:12]

def _n_grip_flips(row) -> int:
    """グリッパ指令の符号反転回数の下限を統計から推定する。
    正常な pick-and-place は 開→閉→開 の2回。
    ⚠️ 統計量からは正確な回数が出ないので、ここでは使わず
       action の std が高いものを候補として見るだけに留める。"""
    return -1

# ⚠️ LeRobotDatasetMetadata.episodes には stats/action/* が無いので、
#    メタデータの parquet を直接取ってくる（約15MB。データ本体は不要）。
_sig = {}
if DEDUP_TRAJECTORIES:
    from huggingface_hub import snapshot_download as _snap
    _meta_dir = Path(run_hf_with_retry(lambda: _snap(
        repo_id=DATASET_REPO, revision=DATASET_REVISION, repo_type="dataset",
        allow_patterns=["meta/episodes/**"], token=False, max_workers=1)))
    _parts = sorted(_meta_dir.rglob("*.parquet"))
    print(f"エピソード表: {len(_parts)} ファイル")
    _ep_df = pd.concat([pd.read_parquet(p) for p in _parts], ignore_index=True)
    _need = [f"stats/action/{k}" for k in ("min", "max", "mean", "std", "count")]
    _missing = [c for c in _need if c not in _ep_df.columns]
    if _missing:
        print(f"⚠️ 必要な列がありません: {_missing}")
        print(f"   利用可能: {[c for c in _ep_df.columns if 'action' in c]}")
        DEDUP_TRAJECTORIES = False
    else:
        _sig = {int(r["episode_index"]): _traj_signature(r)
                for _, r in _ep_df.iterrows()}
        print(f"軌道署名を計算: {len(_sig)} エピソード / "
              f"ユニーク {len(set(_sig.values()))}")

import random as _rnd_mod
_rng_block = _rnd_mod.Random(SEED)

for actual in wanted:
    pool = task_to_episodes[actual]
    if DEDUP_TRAJECTORIES:
        seen, uniq = set(), []
        for e in pool:                       # 若い episode_index を代表に採る
            s = _sig.get(int(e))
            if s is None or s in seen:
                continue
            seen.add(s); uniq.append(e)
        n_before = len(pool)
        pool = uniq
        print(f"  重複排除 {n_before:4d} → {len(pool):4d} 軌道  {actual[:46]}")
    # ★ 2026-08-10: 連続ブロック抽出
    #   libero_plus は観測を動画(mp4)で持つ。全域に散らばった話数を指定すると
    #   動画チャンクをほぼ全部落とすことになり、4時間経っても学習に入らなかった。
    #   摂動タイプは番号順にシャッフルされており、等間隔60本の明度分布は
    #   全在庫とほぼ一致する（実測）。→ 連続でも代表性は同じ。落とす量だけ減る。
    if EPISODE_PICK == "block":
        _p = sorted(pool)
        _n = min(TRAIN_EPISODES_PER_TASK, len(_p))
        _st = _rng_block.randrange(0, max(1, len(_p) - _n + 1))
        selected_by_task[actual] = _p[_st:_st + _n]
    else:
        selected_by_task[actual] = choose_evenly_spaced(pool, TRAIN_EPISODES_PER_TASK)

if missing:
    print("\n❌ 見つからなかったタスク:")
    for task_name in missing:
        print(f"\n  「{task_name}」")
        close = difflib.get_close_matches(
            normalize_task_name(task_name), available_by_normalized.keys(), n=5, cutoff=0.5
        )
        for c in close:
            print(f"      候補: {available_by_normalized[c]}")
    raise RuntimeError(
        "TARGET_TASK_NAMES を上の候補に合わせて修正し、セクション4から流し直してください。"
    )

EPISODE_INDICES = sorted(i for v in selected_by_task.values() for i in v)

print("\n=== 学習に使うエピソード（track1 の4種）===")
t1 = {available_by_normalized.get(normalize_task_name(t)) for t in TARGET_TASK_NAMES}
for task_name, indices in selected_by_task.items():
    if task_name in t1:
        print(f"  {len(indices):4d}話  (在庫 {len(task_to_episodes[task_name]):5d})  {task_name[:66]}")
others = [k for k in selected_by_task if k not in t1]
if others:
    n_oth = sum(len(selected_by_task[k]) for k in others)
    print(f"\n  その他 {len(others)} タスク: 計 {n_oth} 話")
    for k in others[:5]:
        print(f"      {len(selected_by_task[k]):3d}話  {k[:66]}")
    if len(others) > 5:
        print(f"      … 他 {len(others)-5} タスク")

print(f"\n  ★ 合計 {len(EPISODE_INDICES)} 話 / {len(selected_by_task)} タスク")
print("    参考: v1 = 240話/4タスク・128k（公式 0.13178・最良）")
print("          v2 = 画像拡張6種（公式 0.08735）/ v4 = 40タスク種（公式 0.06948）どちらも悪化")

if len(EPISODE_INDICES) > 2000:
    print("\n⚠️ エピソードが多く、データの初回取得に時間がかかります。")
    print("   TRAIN_EPISODES_PER_TASK を下げるか USE_ALL_TASKS=False に。")


## 7. ベース重みを取得する

In [ ]:
BASE_MODEL_LOCAL = cached_or_downloaded_snapshot(
    BASE_MODEL_REPO,
    BASE_MODEL_REVISION,
    ignore_patterns=["eval/*", "*.mp4"],      # 評価動画は不要
)
print(f"ベースモデル: {BASE_MODEL_LOCAL}")
for f in sorted(BASE_MODEL_LOCAL.glob("*")):
    if f.is_file():
        print(f"  {f.stat().st_size/1e6:9.2f} MB  {f.name}")

# ★ ゲート付きリポジトリ名がチェックポイントの JSON に埋まっているので差し替える。
#   これをやらないと lerobot-train が起動直後に 401 GatedRepoError で落ちる。
print("ベース重みのゲート回避:")
patch_gated_tokenizer(BASE_MODEL_LOCAL)


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
OUTPUT_DIR = Path("/content/drive/MyDrive/parc2026/outputs/pi05_libero_t1_lora")
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
SAVE_FREQ = max(500, STEPS // 4)

print("出力先  :", OUTPUT_DIR)
print("保存間隔:", f"{SAVE_FREQ:,} step ごと（{STEPS//SAVE_FREQ} 回）")

## 8. LoRA 学習を実行する

**重要なフラグ**

| フラグ | 意味 |
|---|---|
| `--policy.input_features=null` `--policy.output_features=null` | データセットから入出力の形を作り直す |
| `--policy.empty_cameras=0` | **カメラ2台構成にする**（ベースは3台+空2枚を期待していて、PARCの2台と合わない） |
| `--policy.freeze_vision_encoder=true` | 視覚エンコーダは凍結 |
| `--policy.train_expert_only=true` | action expert だけ学習 |

In [ ]:
import re
from collections import deque
from pathlib import Path

episodes_json = "[" + ",".join(map(str, EPISODE_INDICES)) + "]"

command = [
    "lerobot-train",
    f"--policy.path={BASE_MODEL_LOCAL}",
    "--policy.push_to_hub=false",
    "--policy.repo_id=null",
    "--policy.input_features=null",
    "--policy.output_features=null",
    # ⚠️ pi05 のチェックポイントは画像3枚（image / image2 / empty_camera_0）で
    #    学習されているが、input_features=null にすると
    #    データセット側の front / wrist の2枚から再導出される。
    #    採点側 MyPolicy も front / wrist の2枚を送るので、ここを揃える。
    "--policy.empty_cameras=0",
    "--policy.freeze_vision_encoder=true",
    # ★ PaliGemma-3B 本体を凍結し、gemma_300m の action expert だけを学習する。
    #   3B をまるごと更新するのは Colab の VRAM でもクレジットでも現実的でない。
    "--policy.train_expert_only=true",
    "--policy.compile_model=false",
    f"--policy.optimizer_lr={LEARNING_RATE}",
    f"--policy.scheduler_decay_lr={FINAL_LEARNING_RATE}",
    f"--policy.scheduler_warmup_steps={WARMUP_STEPS}",
    f"--policy.scheduler_decay_steps={STEPS}",
    f"--dataset.repo_id={DATASET_REPO}",
    f"--dataset.revision={DATASET_REVISION}",
    f"--dataset.episodes={episodes_json}",
    "--dataset.use_imagenet_stats=false",
    "--dataset.video_backend=torchcodec",
    # 画像拡張（既定は無効）。
    # ⚠️ 2026-08-09 実測: 6種全部を有効にすると悪化した。
    #    ローカルPARC 95%→85%、RMSジャーク 15.88→20.17(+27%)、姿勢変化量 +26%。
    #    Colab の LeRobot 評価でも ベース90%→70%。独立した2測定が一致。
    #    ジャーク悪化は汎化では説明がつかない＝動きそのものが雑になった。
    #    犯人として疑わしいのは affine（画像と行動の対応をずらす）と
    #    hue（LIBERO は色で物体を識別する）。試すなら3種に絞ること:
    #      "--dataset.image_transforms.enable=true",
    #      "--dataset.image_transforms.tfs.hue.weight=0.0",
    #      "--dataset.image_transforms.tfs.affine.weight=0.0",
    #      "--dataset.image_transforms.tfs.saturation.weight=0.0",
    f"--dataset.image_transforms.enable={'true' if USE_AUGMENTATION else 'false'}",
    f"--output_dir={OUTPUT_DIR}",
    "--job_name=pi05_libero_t1_lora",
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    "--num_workers=2",
    "--persistent_workers=false",
    "--env_eval_freq=0",
    "--eval_steps=0",
    f"--seed={SEED}",
    "--save_checkpoint=true",
    f"--save_freq={SAVE_FREQ}",
    "--save_checkpoint_to_hub=false",
    f"--log_freq={LOG_FREQ}",
    "--wandb.enable=false",
    "--peft.method_type=LORA",
    f"--peft.r={LORA_R}",
    f"--peft.lora_alpha={LORA_ALPHA}",
]

training_env = os.environ.copy()
training_env["PYTHONPATH"] = str(LEROBOT_SRC) + os.pathsep + training_env.get("PYTHONPATH", "")
training_env["ACCELERATE_MIXED_PRECISION"] = MIXED_PRECISION
training_env["PYTHONUNBUFFERED"] = "1"
training_env["HF_HUB_VERBOSITY"] = "warning"
training_env["PYTHONWARNINGS"] = "ignore"

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

print(f"学習開始: {STEPS:,} steps × batch {BATCH_SIZE} = {STEPS*BATCH_SIZE:,} サンプル")
print("（ベース重み 7.5GB とデータの初回ダウンロードが先に走ります）\n")

started = time.time()
process = subprocess.Popen(
    command, cwd=LEROBOT_DIR, env=training_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

recent_lines: deque[str] = deque(maxlen=80)
assert process.stdout is not None

# ★ 2026-08-10 修正: 進捗が一切見えなかった問題
#   旧実装は step 行以外を全部捨てていた。そのため
#   「ダウンロード中なのか固まっているのか」が4時間わからなかった。
#   → 60秒ごとに、キャッシュ容量と直近の出力を必ず出す。
def _cache_size() -> str:
    root = os.environ.get("HF_HOME") or os.path.expanduser("~/.cache/huggingface")
    total = 0
    for p in Path(root).rglob("*"):
        try:
            if p.is_file():
                total += p.stat().st_size
        except OSError:
            pass
    return f"{total/1e9:.2f} GB"


last_beat = time.time()
last_size = None
first_step_seen = False

for raw_line in process.stdout:
    line = raw_line.replace("\r", "").strip()
    if not line:
        continue
    recent_lines.append(line)

    if "step:" in line and "loss:" in line:
        first_step_seen = True
        step_match = re.search(r"step:\s*([0-9KM.]+)", line)
        loss_match = re.search(r"loss:([0-9.eE+-]+)", line)
        lr_match = re.search(r"lr:([0-9.eE+-]+)", line)
        elapsed = time.time() - started
        print(f"  step {step_match.group(1) if step_match else '?':>7}/{STEPS}  "
              f"loss={loss_match.group(1) if loss_match else 'n/a':>9}  "
              f"lr={lr_match.group(1) if lr_match else 'n/a':>9}  "
              f"経過 {elapsed/60:5.1f}分")
        last_beat = time.time()
        continue

    # 学習に入るまでは、60秒ごとに生存確認を出す
    if not first_step_seen and time.time() - last_beat >= 60:
        size = _cache_size()
        stalled = "" if size != last_size else "   ⚠️ 増えていない"
        print(f"  [準備中 {(time.time()-started)/60:5.1f}分]  キャッシュ {size}{stalled}")
        print(f"      直近: {line[:110]}")
        last_size, last_beat = size, time.time()

return_code = process.wait()
if return_code != 0:
    print("\n".join(recent_lines))
    raise RuntimeError(f"Training failed: {return_code}")

print(f"\n学習完了。所要 {(time.time()-started)/60:.1f} 分")

## 9. LoRA をマージしてモデル全体を保存する

In [ ]:
import contextlib
import gc
import io
import json
import os

from peft import PeftModel
from safetensors import safe_open
from lerobot.configs import PreTrainedConfig
import lerobot.policies  # noqa: F401  ← pi05 を draccus に登録させる

def _mem() -> str:
    """CPU RAM と VRAM の使用量。3B のマージはどちらも危ない。"""
    try:
        import psutil
        vm = psutil.virtual_memory()
        cpu = f"RAM {vm.used/2**30:.1f}/{vm.total/2**30:.0f}GB"
    except Exception:
        cpu = "RAM ?"
    if torch.cuda.is_available():
        gpu = (f" / VRAM {torch.cuda.memory_allocated()/2**30:.1f}"
               f"/{torch.cuda.get_device_properties(0).total_memory/2**30:.0f}GB")
    else:
        gpu = ""
    return cpu + gpu


checkpoint_dir = OUTPUT_DIR / "checkpoints" / f"{STEPS:06d}" / "pretrained_model"
if not (checkpoint_dir / "adapter_model.safetensors").is_file():
    found = sorted((OUTPUT_DIR / "checkpoints").glob("*/pretrained_model"))
    raise FileNotFoundError(f"アダプタが見つかりません。存在するのは: {found}")

gc.collect()
torch.cuda.empty_cache()

merge_config = PreTrainedConfig.from_pretrained(checkpoint_dir)
merge_config.pretrained_path = BASE_MODEL_LOCAL
merge_config.use_peft = False

POLICY_TYPE = getattr(merge_config, "type", "smolvla")
if POLICY_TYPE == "pi05":
    from lerobot.policies.pi05.modeling_pi05 import PI05Policy as PolicyCls
elif POLICY_TYPE == "pi0":
    from lerobot.policies.pi0.modeling_pi0 import PI0Policy as PolicyCls
else:
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy as PolicyCls
    # SmolVLA だけが VLM 重みを別途読む。π0.5 は PaliGemma の設定をコード生成し、
    # 重みはチェックポイントから来るので、この2つの属性を持たない。
    merge_config.load_vlm_weights = False
    merge_config.vlm_model_name = VLM_REPO

# ★ 2026-08-10: device="cpu" + float32 だと 3B のマージで CPU RAM を 25GB 前後使う。
#   Colab は CPU RAM を使い切るとランタイムを落とす（アイドル切断ではなく OOM kill）。
#   → 大きいモデルは GPU 上で bfloat16 のままマージする。A100 40GB なら余裕がある。
BIG = POLICY_TYPE in ("pi05", "pi0")
MERGE_DEVICE = "cuda" if (BIG and torch.cuda.is_available()) else "cpu"
merge_config.device = MERGE_DEVICE
if BIG and hasattr(merge_config, "dtype"):
    merge_config.dtype = "bfloat16"     # float32 だと倍のメモリを食う

print(f"policy type   : {POLICY_TYPE}  ({PolicyCls.__name__})")
print(f"マージ先      : {MERGE_DEVICE}"
      + (f" / dtype {getattr(merge_config, 'dtype', '-')}" if BIG else ""))
print(f"開始時        : {_mem()}")

# ⚠️ 出力は握り潰さない。今夜、見えない失敗で何度も止まった。
base_policy = PolicyCls.from_pretrained(BASE_MODEL_LOCAL, config=merge_config, strict=False)
print(f"ベース読込後  : {_mem()}")

peft_policy = PeftModel.from_pretrained(base_policy, checkpoint_dir,
                                        is_trainable=False, torch_device=MERGE_DEVICE)
print(f"アダプタ適用後: {_mem()}")

# safe_merge=True は検算のためにコピーを取る。大きいモデルでは切る。
merged_policy = peft_policy.merge_and_unload(safe_merge=not BIG)
print(f"マージ後      : {_mem()}")

merged_policy = merged_policy.to("cpu")
gc.collect()
torch.cuda.empty_cache()

shutil.rmtree(MERGED_MODEL_DIR, ignore_errors=True)
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

merged_policy.config.use_peft = False
merged_policy.config.pretrained_path = None
merged_policy.config.push_to_hub = False
merged_policy.config.repo_id = None
merged_policy.config.device = None
if not BIG:
    merged_policy.config.load_vlm_weights = False
    merged_policy.config.vlm_model_name = VLM_REPO
merged_policy.save_pretrained(MERGED_MODEL_DIR)
print(f"保存後        : {_mem()}")

for pattern in ["policy_preprocessor.json", "policy_preprocessor*.safetensors",
                "policy_postprocessor.json", "policy_postprocessor*.safetensors"]:
    for source_path in checkpoint_dir.glob(pattern):
        shutil.copy2(source_path, MERGED_MODEL_DIR / source_path.name)

with safe_open(MERGED_MODEL_DIR / "model.safetensors", framework="pt", device="cpu") as w:
    keys = list(w.keys())
    if any("lora_" in k.lower() for k in keys):
        raise RuntimeError("マージ後も LoRA パラメータが残っています。")

merged_cfg = json.loads((MERGED_MODEL_DIR / "config.json").read_text(encoding="utf-8"))
img_feats = [k for k in merged_cfg["input_features"] if "images" in k]
size_gb = sum(f.stat().st_size for f in MERGED_MODEL_DIR.rglob("*") if f.is_file()) / 2**30
print(f"\n重みキー数    : {len(keys)}  / lora_ を含む: 0")
print(f"input_features: {img_feats}")
print(f"empty_cameras : {merged_cfg.get('empty_cameras')}")
print(f"保存サイズ    : {size_gb:.2f} GB")
if len(img_feats) != 2 or merged_cfg.get("empty_cameras") not in (0, None):
    raise RuntimeError("カメラ構成が2台になっていません。PARC の観測と合いません。")
print("2カメラ構成 — PARC の観測と一致")

del peft_policy, base_policy, merged_policy
gc.collect()
torch.cuda.empty_cache()
print(f"解放後        : {_mem()}")

# ★ 評価(セクション12)は zip 化より前に MERGED_MODEL_DIR を読む。
#   ゲート付きリポジトリ名をここで潰しておかないと同じ 401 で落ちる。
print("\nマージ済み重みのゲート回避:")
patch_gated_tokenizer(MERGED_MODEL_DIR)


In [ ]:
import shutil
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
dst = Path("/content/drive/MyDrive/parc2026/pi05_v2_merged")
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(MERGED_MODEL_DIR, dst, dirs_exist_ok=True)
print("退避:", dst, f"{sum(f.stat().st_size for f in dst.rglob('*') if f.is_file())/2**30:.2f} GB")

In [ ]:
# ⚠️ トークンを平文で書かない。ノートブックは Colab・Downloads・共有先に残る。
#    以前ここに書いたものは huggingface.co/settings/tokens で必ず Revoke すること。
from getpass import getpass

from huggingface_hub import HfApi

REPO = "a2taka-sourse/pi05-parc-t1"
_token = getpass("parc")

api = HfApi(token=_token)
api.create_repo(REPO, private=True, exist_ok=True)      # ★ 必ず private
api.upload_folder(folder_path=str(MERGED_MODEL_DIR), repo_id=REPO)
print("完了:", REPO)
del _token


---

## 終わったら

1. `pi05_t1_lora_merged.zip`（**約7.5GB**）を回収する。
   ⚠️ ブラウザ経由のダウンロードは失敗しやすい。**Google Drive にマウントして書き出すほうが確実。**

2. 母艦で提出zipを作る:

   ```bash
   cd C:/Users/odeod/dev/PARC2026_pre
   # zip を submission_template/model_weights_pi05/ に展開してから
   python build_zip.py model_weights_pi05 submission_pi05.zip
   ```

   `policy_server.py` は `config.json` の `type` を見て SmolVLA / π0.5 を自動で切り替える。
   **コードの書き換えは不要。** PaliGemma のトークナイザは
   `submission_template/vlm_pi05/` に同梱済み。

3. ⚠️ **提出前に、セクション12の結果を必ず見ること。**
   ローカルでは測れないので、ここが唯一の判断材料。

   | 判断 | 行動 |
   |---|---|
   | π0.5 が SmolVLA(v1) のローカル成功率 0.95 を**明確に上回る** | 提出する |
   | 同等かそれ以下 | **提出しない**。0.13178 を守る |

   ⚠️ SmolVLA での実績上、**1〜2話の差は公式スコアと相関しなかった**（2回とも外した）。
   大きく動いたときだけ信じること。

## 提出枠の使い方（1日1回・採点完了時点でカウント）

| 日 | 提出 |
|---|---|
| 8/10 | v6（重複排除版）— 実験枠。ローカルでは負けているので情報目的のみ |
| 8/11〜8/13 | π0.5（セクション12が良かった場合のみ） |
| **8/14** | **その時点で最良の構成を必ず出し直して確定する** |


## 15. 汎化テスト — **PARC の採点器で未知24変種を測る**

⚠️ 既存の比較評価（セクション12）は π0.5 では 100% に飽和し、**1mm 衝突ゲートも入っていない**ため、推論パラメータの比較に使えない。

**事前に `policy_server.py` を `/content/` へアップロードすること。**


In [ ]:
# ★ pi_gemma.py のパッチをやり直す
#   ⚠️ 前のパッチは import ブロックの内側に定義を挿し込んでいて IndentationError になった。
#      git で元に戻してから、import ブロックの外へ入れ直す。最後に compile で検証する。
import inspect
import subprocess
from pathlib import Path

_pg = LEROBOT_SRC / "lerobot" / "policies" / "pi_gemma.py"
subprocess.run(["git", "checkout", "--", "src/lerobot/policies/pi_gemma.py"],
               cwd=str(LEROBOT_DIR), check=True)
t = _pg.read_text(encoding="utf-8")
assert "_ACCEPTS_CACHE_POSITION" not in t, "元に戻っていない"

# ① import ブロックの外（else 節の直後）に判定を置く
anchor = "    create_causal_mask = None\n"
assert anchor in t, "アンカーが見つからない"
t = t.replace(anchor, anchor + """

# transformers 5.14.1 では create_causal_mask から cache_position が削除された。
# 5.5.4 でも "not used anymore but kept for BC" ＝渡さなくても結果は同じ。
import inspect as _inspect

_ACCEPTS_CACHE_POSITION = bool(
    create_causal_mask is not None
    and "cache_position" in _inspect.signature(create_causal_mask).parameters
)
""", 1)

# ② 呼び出し側を条件付きに
old = ("            cache_position=cache_position,\n"
       "            past_key_values=past_key_values,\n"
       "            position_ids=position_ids,\n"
       "        )")
new = ("            past_key_values=past_key_values,\n"
       "            position_ids=position_ids,\n"
       "            **({\"cache_position\": cache_position}\n"
       "               if _ACCEPTS_CACHE_POSITION else {}),\n"
       "        )")
assert old in t, "呼び出し箇所が見つからない"
t = t.replace(old, new, 1)

_pg.write_text(t, encoding="utf-8")
compile(t, str(_pg), "exec")          # ★ これを前回やっていなかった
print("pi_gemma: 修正 / 構文OK")

import transformers
print("transformers:", transformers.__version__)


In [ ]:
import shutil
from pathlib import Path

dst = Path("/content/vlm_pi05")
if not dst.is_dir():
    shutil.copytree(TOKENIZER_DIR, dst)
print("同梱物:", sorted(p.name for p in dst.iterdir()))


In [ ]:
# ★ 診断: サーバーを立てて1タスクだけ回し、出力を全部見る
import os, signal, subprocess, time
from pathlib import Path
import requests

SERVER = Path("/content/policy_server.py")
BASE = Path(BASE_MODEL_LOCAL)
TASK = HOLDOUT["libero_spatial"][3]      # light_33

env = os.environ.copy()
env["PARC_MODEL_DIR"] = str(BASE)
env["PYTHONPATH"] = str(LEROBOT_SRC)
env["PYTHONUNBUFFERED"] = "1"
log = Path("/content/_diag_server.log")
proc = subprocess.Popen(["python", str(SERVER), "--port", "8000"],
                        stdout=log.open("w"), stderr=subprocess.STDOUT, env=env)
print("サーバー起動待ち", end="", flush=True)
for _ in range(200):
    try:
        if requests.get("http://localhost:8000/health", timeout=3).ok: break
    except Exception: pass
    if proc.poll() is not None:
        print("\n落ちた:\n" + log.read_text()[-2000:]); raise SystemExit
    time.sleep(3); print(".", end="", flush=True)
print(" OK")

PIPE_ENV = os.environ.copy()
PIPE_ENV["PYTHONPATH"] = os.pathsep.join(
    [str(LIBERO_PLUS_DIR), str(LEROBOT_SRC), PIPE_ENV.get("PYTHONPATH", "")])
PIPE_ENV["MUJOCO_GL"] = "egl"
PIPE_ENV["PYOPENGL_PLATFORM"] = "egl"

try:
    r = subprocess.run(
        ["python", "-m", "pipeline", "--server-url", "http://localhost:8000",
         "--track", "track1", "--benchmark", "libero_spatial", "--tasks", TASK,
         "--timeout", "600", "--n-episodes", "1", "--max-steps", "600",
         "--output-dir", "/content/_diag1"],
        cwd=str(PARC_DIR), env=PIPE_ENV, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
finally:
    proc.send_signal(signal.SIGINT); time.sleep(2); proc.kill()

print("終了コード:", r.returncode)
print("=" * 70)
print(r.stdout)                      # ★ 全部
print("=" * 70)
print("サーバー側ログ:")
print(log.read_text()[-2000:])


In [ ]:
# ★ PARC の本物の採点器を Colab に持ち込み、未知の摂動24件で測る
#
#   なぜ要るか:
#     π0.5(3B) は母艦(RAM 4GB / モデル 8.9GB)で動かせない。
#     このノートブックの既存評価は「4タスク・成功率のみ」で、
#       ① π0.5 では 100% に張り付いて差が出ない（飽和）
#       ② PARC の成功判定にある **1mm 衝突ゲート** が入っていない
#     ため、推論パラメータの比較に使えない。
#
#   PARC の pipeline は公開リポジトリにあり 144KB、依存は libero / msgpack / requests だけ。
#   24件（摂動タイプで層化: light5 table4 add8 tb3 level3 moved_level1）を
#   **公式と同じ N=1** で回す。SmolVLA v1 の実測は 14/24、ベース重みは 13/24。
#
# ⚠️ 事前準備（1回だけ）
#     母艦の  C:\Users\odeod\dev\PARC2026_pre\submission_template\policy_server.py
#     を Colab の左パネルから /content/ にアップロードしておくこと。
#     （clone した公開リポジトリの policy_server.py は元のテンプレートで、
#       私たちの MyPolicy ではない。それを使うと別物を測ることになる）
import json
import os
import re
import shutil
import signal
import subprocess
import time
from collections import defaultdict
from pathlib import Path

import requests

PARC_DIR = Path("/content/PARC2026_pre")
SERVER = Path("/content/policy_server.py")          # ← アップロードしたもの
# ★ 2モデルを同じ 24 件で比べる。
#   ⚠️ ベース＝公開重みそのもの。前回の LoRA はベースと差が出なかった
#      （4タスク 100% vs 100%）ので、ベースは公式 0.36363 の良い代理になる。
#   学習が効いたかは「学習後 − ベース」で読む。
EVAL_MODELS = {
    "ベース": Path(BASE_MODEL_LOCAL),
    "学習後": Path(MERGED_MODEL_DIR),
}

# 推論設定。★ ここを変えて比較する
PARC_ENV = {
    "PARC_N_ACTION_STEPS": "10",     # SmolVLA で決めた値。π0.5 では未検証
    "PARC_SMOOTH": "0.5",            # 同上
    # "PARC_NUM_STEPS": "20",        # flow matching の逆拡散回数（π0.5 既定 10）
    # "PARC_VOTE_K": "3",
}

HOLDOUT = {
    "libero_spatial": [
        "pick_up_the_black_bowl_in_the_top_drawer_of_the_wooden_cabinet_and_place_it_on_the_plate_add_17",
        "pick_up_the_black_bowl_in_the_top_drawer_of_the_wooden_cabinet_and_place_it_on_the_plate_add_24",
        "pick_up_the_black_bowl_in_the_top_drawer_of_the_wooden_cabinet_and_place_it_on_the_plate_level4_sample3",
        "pick_up_the_black_bowl_in_the_top_drawer_of_the_wooden_cabinet_and_place_it_on_the_plate_light_33",
        "pick_up_the_black_bowl_in_the_top_drawer_of_the_wooden_cabinet_and_place_it_on_the_plate_table_8",
        "pick_up_the_black_bowl_in_the_top_drawer_of_the_wooden_cabinet_and_place_it_on_the_plate_tb_9",
    ],
    "libero_object": [
        "pick_up_the_tomato_sauce_and_place_it_in_the_basket_add_20",
        "pick_up_the_tomato_sauce_and_place_it_in_the_basket_add_30",
        "pick_up_the_tomato_sauce_and_place_it_in_the_basket_level1_sample1",
        "pick_up_the_tomato_sauce_and_place_it_in_the_basket_light_13",
        "pick_up_the_tomato_sauce_and_place_it_in_the_basket_table_4",
        "pick_up_the_tomato_sauce_and_place_it_in_the_basket_tb_6",
        "pick_up_the_milk_and_place_it_in_the_basket_add_11",
        "pick_up_the_milk_and_place_it_in_the_basket_add_18",
        "pick_up_the_milk_and_place_it_in_the_basket_level2_sample1",
        "pick_up_the_milk_and_place_it_in_the_basket_light_37",
        "pick_up_the_milk_and_place_it_in_the_basket_table_12",
        "pick_up_the_milk_and_place_it_in_the_basket_tb_11",
    ],
    "libero_goal": [
        "put_the_bowl_on_the_stove_add_14",
        "put_the_bowl_on_the_stove_add_27",
        "put_the_bowl_on_the_stove_light_24",
        "put_the_bowl_on_the_stove_light_37",
        "put_the_bowl_on_the_stove_moved_level2_sample4",
        "put_the_bowl_on_the_stove_table_24",
    ],
}
REF_SMOLVLA = {"light": "5/5", "level": "2/3", "tb": "2/3", "add": "4/8",
               "table": "1/4", "moved_level": "0/1"}

assert SERVER.is_file(), (
    f"{SERVER} が無い。母艦の submission_template/policy_server.py を "
    "左パネルから /content/ にアップロードすること"
)

# ---------- ① 採点器を取得 ----------
if not (PARC_DIR / "pipeline").is_dir():
    subprocess.run(["git", "clone", "--quiet", "--depth", "1",
                    "https://github.com/matsuolab/PARC2026_pre.git", str(PARC_DIR)],
                   check=True)
print("採点器  :", PARC_DIR / "pipeline")

# ---------- ② Colab 側 LeRobot に、提出物と同じ修正を当てる ----------
#   ⚠️ 母艦では vendor/lerobot/policies/pi_gemma.py を直したが、
#      Colab の LeRobot は別物なので同じ修正が要る。
#      当てないと transformers の版によっては
#        TypeError: create_causal_mask() got an unexpected keyword argument 'cache_position'
#      で落ちる（公式スコア 0.000 の原因）。
_pg = LEROBOT_SRC / "lerobot" / "policies" / "pi_gemma.py"
_t = _pg.read_text(encoding="utf-8")
if "_ACCEPTS_CACHE_POSITION" not in _t:
    _t = _t.replace(
        "            cache_position=cache_position,\n"
        "            past_key_values=past_key_values,\n"
        "            position_ids=position_ids,\n"
        "        )",
        "            past_key_values=past_key_values,\n"
        "            position_ids=position_ids,\n"
        "        ) if not _ACCEPTS_CACHE_POSITION else create_causal_mask(\n"
        "            config=self.config,\n"
        "            inputs_embeds=inputs_embeds,\n"
        "            attention_mask=attention_mask,\n"
        "            cache_position=cache_position,\n"
        "            past_key_values=past_key_values,\n"
        "            position_ids=position_ids,\n"
        "        )", 1)
    _t = _t.replace(
        "from transformers.masking_utils import create_causal_mask",
        "from transformers.masking_utils import create_causal_mask\n"
        "import inspect as _inspect\n"
        "_ACCEPTS_CACHE_POSITION = 'cache_position' in _inspect.signature("
        "create_causal_mask).parameters", 1)
    _pg.write_text(_t, encoding="utf-8")
    print("pi_gemma: バージョン非依存に修正")
else:
    print("pi_gemma: 修正済み")

# ---------- ③ ポリシーサーバーを起動 ----------
def start_server(model_dir: Path):
    env = os.environ.copy()
    env.update(PARC_ENV)
    env["PARC_MODEL_DIR"] = str(model_dir)
    env["PYTHONPATH"] = str(LEROBOT_SRC)
    env["PYTHONUNBUFFERED"] = "1"
    log = Path("/content/parc_server.log")
    proc = subprocess.Popen(["python", str(SERVER), "--port", "8000"],
                            stdout=log.open("w"), stderr=subprocess.STDOUT, env=env)
    print("   起動待ち", end="", flush=True)
    for _ in range(200):
        try:
            if requests.get("http://localhost:8000/health", timeout=3).ok:
                break
        except Exception:
            pass
        if proc.poll() is not None:
            print("\n--- 落ちた ---\n" + log.read_text()[-3000:])
            raise RuntimeError("policy_server.py が起動しなかった")
        time.sleep(3)
        print(".", end="", flush=True)
    print(" OK")
    for line in log.read_text().splitlines():
        if "policy type" in line or "n_action_steps" in line or "平滑" in line:
            print("     ", line)
    return proc


# ---------- ④ 24件 ----------
ROOT = Path("/content/parc_holdout")
shutil.rmtree(ROOT, ignore_errors=True)
t0 = time.time()
for label, model_dir in EVAL_MODELS.items():
    print(f"\n{'='*64}\n■ {label}   {model_dir}\n{'='*64}", flush=True)
    server = start_server(model_dir)
    try:
        for suite, names in HOLDOUT.items():
            print(f"  ▼ {suite}  ({len(names)} タスク × 1話)", flush=True)
            r = subprocess.run(
                ["python", "-m", "pipeline", "--server-url", "http://localhost:8000",
                 "--track", "track1", "--benchmark", suite, "--tasks", *names,
                 "--timeout", "10800", "--n-episodes", "1", "--max-steps", "600",
                 "--output-dir", str(ROOT / label / suite)],
                cwd=str(PARC_DIR), text=True,
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            for line in r.stdout.splitlines():
                if "タスク評価完了" in line or "総合スコア" in line:
                    print("     ", line.strip(), flush=True)
    finally:
        server.send_signal(signal.SIGINT)
        time.sleep(2)
        server.kill()
        time.sleep(3)

# ---------- ⑤ 集計 ----------
def kind(n):
    m = re.search(r"_(moved_level|level|light|table|tb|add)_?\d*", n)
    return m.group(1) if m else "?"


res = {}
for label in EVAL_MODELS:
    d = {}
    for f in (ROOT / label).rglob("*.json"):
        for tr in json.loads(f.read_text()).get("tracks", []):
            for t in tr.get("tasks", []):
                d[t["task_name"]] = (t.get("success_rate"),
                                     (t.get("metrics") or {}).get("collision_rate"))
    res[label] = d

labels = list(EVAL_MODELS)
names = sorted(set().union(*[set(d) for d in res.values()]))
by = {l: defaultdict(list) for l in labels}
for n in names:
    for l in labels:
        by[l][kind(n)].append(res[l].get(n, (0, None))[0])

print(f"\n{'='*68}")
print(f"汎化テスト（未知24変種・N=1）  所要 {(time.time()-t0)/60:.1f}分")
print(f"{'='*68}")
print(f"  {'摂動':<14}" + "".join(f"{l:>11}" for l in labels) + "   SmolVLA v1")
first = labels[0]
for k in sorted(by[first], key=lambda k: -sum(by[first][k]) / len(by[first][k])):
    cells = "".join(f"{sum(by[l][k]):.0f}/{len(by[l][k]):<8}" for l in labels)
    print(f"  {k:<14}{cells}   {REF_SMOLVLA.get(k, '-')}")

print()
for l in labels:
    v = [res[l].get(n, (0, None))[0] for n in names]
    print(f"  ★ {l:<8} {sum(v):.0f}/{len(v)}  ({sum(v) / len(v) * 100:.1f}%)")
print("     参考: SmolVLA v1 = 14/24 (58.3%) → 公式 0.13178")
print("           π0.5 ベース相当 → 公式 0.36363")

print("\n  結果が割れた個体:")
n_diff = 0
for n in names:
    a = res[labels[0]].get(n, (0, None))[0]
    b = res[labels[-1]].get(n, (0, None))[0]
    if a != b:
        n_diff += 1
        who = "学習後のみ成功" if b > a else "ベースのみ成功"
        print(f"    {who}  {kind(n):<12} {n[-42:]}")
if n_diff == 0:
    print("    なし ＝ 学習は挙動を変えていない（前回の LoRA と同じ）")

print("\n  ⚠️ 読み方: 複数の摂動タイプが同じ向きに大きく動いたときだけ信じる。")
print("     1〜2件の差は公式スコアと相関しなかった（SmolVLA で2回とも外した）。")


In [ ]:
# ★ 1タスクだけ、出力を全部見る
import subprocess
r = subprocess.run(
    ["python", "-m", "pipeline", "--server-url", "http://localhost:8000",
     "--track", "track1", "--benchmark", "libero_spatial",
     "--tasks", HOLDOUT["libero_spatial"][3],      # light_33（ベースが成功したはずの1件）
     "--timeout", "600", "--n-episodes", "1", "--max-steps", "600",
     "--output-dir", "/content/_diag1"],
    cwd=str(PARC_DIR), env=PIPE_ENV, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print("終了コード:", r.returncode)
print(r.stdout[-8000:])


In [ ]:
# ★ 自己完結の診断（PIPE_ENV をここで作る）
import os, subprocess
from pathlib import Path

PIPE_ENV = os.environ.copy()
PIPE_ENV["PYTHONPATH"] = os.pathsep.join(
    [str(LIBERO_PLUS_DIR), str(LEROBOT_SRC), PIPE_ENV.get("PYTHONPATH", "")])
PIPE_ENV["MUJOCO_GL"] = "egl"
PIPE_ENV["PYOPENGL_PLATFORM"] = "egl"
PIPE_ENV["PYTHONWARNINGS"] = "ignore"

r = subprocess.run(
    ["python", "-m", "pipeline", "--server-url", "http://localhost:8000",
     "--track", "track1", "--benchmark", "libero_spatial",
     "--tasks", HOLDOUT["libero_spatial"][3],
     "--timeout", "600", "--n-episodes", "1", "--max-steps", "600",
     "--output-dir", "/content/_diag1"],
    cwd=str(PARC_DIR), env=PIPE_ENV, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print("終了コード:", r.returncode)
print(r.stdout[-8000:])


In [ ]:
# ★ LIBERO の torch.load を weights_only=False にする
#   ⚠️ PyTorch 2.6+ の既定 weights_only=True では
#      numpy.core.multiarray._reconstruct が読めず、初期状態が1件も
#      ロードできない → 評価対象0件 → 総合スコア 0.000 になる。
#   ⚠️ weights_only=False は任意コード実行を許すが、読むのは LIBERO 自身の
#      init_files（既知のリポジトリ由来）なので許容する。
import re
from pathlib import Path

n_files = 0
for p in Path(LIBERO_PLUS_DIR).rglob("*.py"):
    try:
        t = p.read_text(encoding="utf-8")
    except (OSError, UnicodeDecodeError):
        continue
    if "torch.load(" not in t or "weights_only" in t:
        continue
    t2 = re.sub(r"torch\.load\(([^)]*?)\)", r"torch.load(\1, weights_only=False)", t)
    if t2 != t:
        p.write_text(t2, encoding="utf-8")
        compile(t2, str(p), "exec")          # ★ 構文検証
        n_files += 1
        print("  修正:", p.relative_to(LIBERO_PLUS_DIR))
print(f"\n{n_files} ファイルを修正")

# カーネル側の読み込み済みモジュールを捨てる
import sys, importlib
for m in list(sys.modules):
    if m.split(".")[0] in {"libero", "robosuite"}:
        del sys.modules[m]
importlib.invalidate_caches()
print("⚠️ このあと セクション15 を実行してください")


In [ ]:
import contextlib, io, os, sys
sys.path.insert(0, str(PARC_DIR))
from pipeline.config import EvalConfig
from pipeline.environment import EnvironmentManager
m = EnvironmentManager(EvalConfig())
with contextlib.redirect_stdout(io.StringIO()):
    infos = {t.name: t for t in m.get_task_infos("libero_spatial")}
print("登録タスク数:", len(infos))
for n in HOLDOUT["libero_spatial"]:
    t = infos.get(n)
    print(f"  {'★' if t and os.path.isfile(t.bddl_file) else '✗'} {n[-40:]}")


In [ ]:
# ① libero がどこにあるか
import libero, os, sys
from pathlib import Path
LIBERO_PKG = Path(libero.__file__).parent.parent      # libero/ の親
print("libero :", libero.__file__)
print("追加すべきパス:", LIBERO_PKG)

# ② robosuite なども確認
for m in ("robosuite", "bddl", "mujoco"):
    try:
        mod = __import__(m); print(f"  OK  {m}: {mod.__file__}")
    except Exception as e:
        print(f"  ✗   {m}: {e}")

# ③ サブプロセス用の環境を作る
PIPE_ENV = os.environ.copy()
PIPE_ENV["PYTHONPATH"] = os.pathsep.join(
    [str(LIBERO_PKG), str(LEROBOT_SRC), PIPE_ENV.get("PYTHONPATH", "")])
PIPE_ENV["MUJOCO_GL"] = "egl"          # Colab はヘッドレス。オフスクリーン描画に必要
PIPE_ENV["PYOPENGL_PLATFORM"] = "egl"
print("\nPYTHONPATH:", PIPE_ENV["PYTHONPATH"])

# ④ 1スイートだけ試す（出力は全部見る）
import subprocess
r = subprocess.run(
    ["python", "-m", "pipeline", "--server-url", "http://localhost:8000",
     "--track", "track1", "--benchmark", "libero_spatial",
     "--tasks", *HOLDOUT["libero_spatial"],
     "--timeout", "600", "--n-episodes", "1", "--max-steps", "600",
     "--output-dir", "/content/_diag"],
    cwd=str(PARC_DIR), env=PIPE_ENV, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print("終了コード:", r.returncode)
print(r.stdout[-6000:])


In [ ]:
baseline_config = PreTrainedConfig.from_pretrained(MERGED_MODEL_DIR)
baseline_config.device = "cpu"
baseline_config.pretrained_path = BASE_MODEL_LOCAL
baseline_config.use_peft = False
baseline_config.load_vlm_weights = False

quiet = io.StringIO()
with contextlib.redirect_stdout(quiet), contextlib.redirect_stderr(quiet):
    baseline_policy = PolicyCls.from_pretrained(
        BASE_MODEL_LOCAL, config=baseline_config, strict=False
    )
if POLICY_TYPE not in ("pi05", "pi0"):     # π0.5 はこの属性を持たない
    baseline_policy.config.vlm_model_name = VLM_REPO
shutil.rmtree(BASELINE_MODEL_DIR, ignore_errors=True)
BASELINE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

baseline_policy.config.use_peft = False
baseline_policy.config.pretrained_path = None
baseline_policy.config.push_to_hub = False
baseline_policy.config.repo_id = None
baseline_policy.config.device = None
baseline_policy.config.load_vlm_weights = False
baseline_policy.config.vlm_model_name = VLM_REPO
baseline_policy.save_pretrained(BASELINE_MODEL_DIR)

for pattern in ["policy_preprocessor.json", "policy_preprocessor*.safetensors",
                "policy_postprocessor.json", "policy_postprocessor*.safetensors"]:
    for source_path in MERGED_MODEL_DIR.glob(pattern):
        shutil.copy2(source_path, BASELINE_MODEL_DIR / source_path.name)

del baseline_policy
gc.collect(); torch.cuda.empty_cache()
print("Baseline ready.（LoRA なし・カメラ構成は学習後と同じ）")

# ★ 評価も同じ JSON 経路を通るので、ここでも差し替える
patch_gated_tokenizer(BASELINE_MODEL_DIR)


In [ ]:
from huggingface_hub import hf_hub_download

LIBERO_PLUS_SHA = "4976dc3"
LIBERO_PLUS_DIR = Path("/content/LIBERO-plus")
LIBERO_PLUS_PACKAGE_ROOT = LIBERO_PLUS_DIR / "libero" / "libero"
LIBERO_PLUS_ASSETS_DIR = LIBERO_PLUS_PACKAGE_ROOT / "assets"

os.environ["MUJOCO_GL"] = "egl"

run_quiet([sys.executable, "-m", "pip", "uninstall", "-y",
           "hf-libero", "libero", "robosuite"], check=False)
run_quiet([sys.executable, "-m", "pip", "install", "--quiet",
           "robosuite==1.4.1", "bddl==1.0.1", "easydict==1.13", "mujoco==3.7.0",
           "matplotlib==3.10.8", "Wand==0.6.13", "scikit-image==0.25.2", "gym==0.26.2"])

if importlib.metadata.version("robosuite") != "1.4.1":
    raise RuntimeError("robosuite 1.4.1 が必要です。")

if not (LIBERO_PLUS_DIR / ".git").is_dir():
    shutil.rmtree(LIBERO_PLUS_DIR, ignore_errors=True)
    run_quiet(["git", "clone", "--quiet",
               "https://github.com/sylvestf/LIBERO-plus.git", str(LIBERO_PLUS_DIR)])

checkout = run_quiet(["git", "-C", str(LIBERO_PLUS_DIR), "checkout", "--quiet",
                      LIBERO_PLUS_SHA], check=False)
if checkout.returncode != 0:
    run_quiet(["git", "-C", str(LIBERO_PLUS_DIR), "fetch", "--quiet",
               "--depth", "1", "origin", LIBERO_PLUS_SHA])
    run_quiet(["git", "-C", str(LIBERO_PLUS_DIR), "checkout", "--quiet", LIBERO_PLUS_SHA])

run_quiet([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
           "-e", str(LIBERO_PLUS_DIR)])

if not LIBERO_PLUS_ASSETS_DIR.is_dir():
    assets_root = Path("/content/libero_plus_assets")
    archive_path = Path(run_hf_with_retry(lambda: hf_hub_download(
        repo_id="Sylvest/LIBERO-plus", repo_type="dataset",
        filename="assets.zip", local_dir=assets_root, token=False,
    )))
    extract_dir = assets_root / "extract"
    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)
    run_quiet(["unzip", "-q", str(archive_path), "-d", str(extract_dir)])
    candidates = sorted([p for p in extract_dir.rglob("assets") if p.is_dir()],
                        key=lambda p: len(p.parts))
    if not candidates:
        raise FileNotFoundError("LIBERO-plus assets not found.")
    LIBERO_PLUS_ASSETS_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(candidates[0]), str(LIBERO_PLUS_ASSETS_DIR))
    shutil.rmtree(assets_root, ignore_errors=True)

libero_config_dir = Path.home() / ".libero"
libero_config_dir.mkdir(parents=True, exist_ok=True)
(libero_config_dir / "config.yaml").write_text("\n".join([
    f"assets: {LIBERO_PLUS_ASSETS_DIR}",
    f"bddl_files: {LIBERO_PLUS_PACKAGE_ROOT / 'bddl_files'}",
    f"datasets: {LIBERO_PLUS_PACKAGE_ROOT.parent / 'datasets'}",
    f"init_states: {LIBERO_PLUS_PACKAGE_ROOT / 'init_files'}",
]) + "\n", encoding="utf-8")

# 評価スクリプトに進捗表示を仕込む
eval_script = LEROBOT_SRC / "lerobot" / "scripts" / "lerobot_eval.py"
source = eval_script.read_text(encoding="utf-8")
source = source.replace("logging.info(pformat(asdict(cfg)))", "logging.debug(pformat(asdict(cfg)))", 1)
source = source.replace("max_episodes_rendered = 0 if cfg.eval.recording else 10",
                        "max_episodes_rendered = 0", 1)
source = source.replace("disable=inside_slurm()", "disable=True")

progress_state = '_EVAL_PROGRESS = {"task_index": 0, "task_total": 0}'
if progress_state not in source:
    anchor = "from tqdm import trange\n"
    if anchor not in source:
        raise RuntimeError("進捗フックの挿入位置が見つかりません。")
    source = source.replace(anchor, anchor + "\n" + progress_state + "\n", 1)

task_loop_anchor = "        for i, (task_group, task_id, env) in enumerate(tasks):\n"
if '_EVAL_PROGRESS["task_index"] = i + 1' not in source:
    if task_loop_anchor not in source:
        raise RuntimeError("タスクループの挿入位置が見つかりません。")
    source = source.replace(task_loop_anchor, task_loop_anchor
        + '            _EVAL_PROGRESS["task_index"] = i + 1\n'
        + '            _EVAL_PROGRESS["task_total"] = len(tasks)\n', 1)

episode_loop_anchor = "    for batch_ix in progbar:\n"
if "EVAL_PROGRESS task=" not in source:
    if episode_loop_anchor not in source:
        raise RuntimeError("エピソードループの挿入位置が見つかりません。")
    source = source.replace(episode_loop_anchor, episode_loop_anchor
        + '        print(f"EVAL_PROGRESS '
        + "task={_EVAL_PROGRESS['task_index']}/{_EVAL_PROGRESS['task_total']} "
        + 'episode={batch_ix + 1}/{n_batches}", flush=True)\n', 1)

eval_script.write_text(source, encoding="utf-8")

libero_plus_path = str(LIBERO_PLUS_DIR)
sys.path = [p for p in sys.path if p != libero_plus_path]
sys.path.insert(0, libero_plus_path)
for module_name in list(sys.modules):
    if module_name.split(".")[0] in {"libero", "robosuite"}:
        del sys.modules[module_name]
importlib.invalidate_caches()

import libero
from libero.libero import benchmark

search_paths = [Path(p).resolve() for p in getattr(libero, "__path__", [])]
if not any(LIBERO_PLUS_DIR.resolve() in p.parents or p == LIBERO_PLUS_DIR.resolve()
           for p in search_paths):
    raise RuntimeError("LIBERO-plus fork が読み込まれていません。")
if LIBERO_PLUS_DIR.resolve() not in Path(benchmark.__file__).resolve().parents:
    raise RuntimeError("LIBERO-plus benchmark が読み込まれていません。")

print("LIBERO-plus ready.")